In [0]:
# To reduce the bias of transactions amount, I will create a log feature of transactions amounts

from pyspark.sql.functions import log1p, hour, dayofweek, avg, when

from pyspark.sql.window import Window

transactions = spark.table("workspace.fraud_unified.unified_fraud_transactions")
transactions = transactions.withColumn(
    "log_amount",
    log1p("amount")
)

# Now, for balance, I will set a ratio based on the amounts and transaction balance to help detecting abnormal spending

transactions = transactions.withColumn(
    "amount_balance_ratio",
    when(transactions.account_balance != 0, transactions.amount / transactions.account_balance).otherwise(None)
)

# Now, timing features are defined, in this case I will use day of week and hour to also identify timely transactions patterns

transactions = transactions.withColumn(
    "transaction_hour",
    hour("transaction_timestamp")
)

transactions = transactions.withColumn(
    "transaction_dayofweek",
    dayofweek("transaction_timestamp")
)

# Now, merchant categories with high risk are defined, to set one high risk feature by merchant

merchant_risk_rate = transactions.groupBy("merchant_category")\
    .agg(avg("is_fraud").alias("merchant_fraud_rate"))

transactions = transactions.join(
    merchant_risk_rate,
    on="merchant_category",
    how="left"
)

# Finally, I will create a feature that captures the average of user transactions, to capture the user spending patterns, but with Window to validate the previous transactions only. This adds a higher level of bias, but it is a good way to capture the user spending patterns.

customer_window = Window.partitionBy("customer_id")\
    .orderBy("transaction_timestamp")\
    .rowsBetween(Window.unboundedPreceding, -1)

transactions = transactions.withColumn(
    "customer_avg_transaction_amount",
    avg("amount").over(customer_window)
)


transaction_features = transactions.select(
    "merchant_category",
    "customer_id",
    "log_amount",
    "amount_balance_ratio",
    "transaction_hour",
    "transaction_dayofweek",
    "device_type",
    "merchant_fraud_rate",
    "channel",
    "location_city",
    "transaction_id",
    "customer_avg_transaction_amount",
    "amount",
    "account_balance"
)

transaction_features.write.format("delta")\
    .mode("overwrite").option("mergeSchema", "true")\
    .saveAsTable("ml_layer.transaction_features")